# FurniCraft ML Model Training & Analysis

This notebook contains machine learning model training and data analysis for the FurniCraft e-commerce platform.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import json
import warnings
warnings.filterwarnings('ignore')

# Set up plotting
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

## 1. Data Loading and Exploration

In [ ]:
# Load the furniture dataset
df = pd.read_csv('../backend/data/clean_products.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

In [ ]:
# Basic data exploration
print("Data Info:")
print(df.info())

print("\nMissing Values:")
missing_data = df.isnull().sum()
missing_percentage = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_data,
    'Missing Percentage': missing_percentage
})
print(missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False))

## 2. Price Analysis

In [ ]:
# Price distribution analysis
price_data = df['price_num'].dropna()

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Histogram
axes[0, 0].hist(price_data, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Price Distribution')
axes[0, 0].set_xlabel('Price ($)')
axes[0, 0].set_ylabel('Frequency')

# Box plot
axes[0, 1].boxplot(price_data, vert=True)
axes[0, 1].set_title('Price Box Plot')
axes[0, 1].set_ylabel('Price ($)')

# Log scale histogram
axes[1, 0].hist(np.log1p(price_data), bins=50, alpha=0.7, color='lightgreen', edgecolor='black')
axes[1, 0].set_title('Log Price Distribution')
axes[1, 0].set_xlabel('Log(Price + 1)')
axes[1, 0].set_ylabel('Frequency')

# Price vs Index scatter
sample_indices = np.random.choice(len(price_data), min(1000, len(price_data)), replace=False)
axes[1, 1].scatter(sample_indices, price_data.iloc[sample_indices], alpha=0.6, color='coral')
axes[1, 1].set_title('Price Scatter Plot (Sample)')
axes[1, 1].set_xlabel('Index')
axes[1, 1].set_ylabel('Price ($)')

plt.tight_layout()
plt.show()

print(f"Price Statistics:")
print(f"Mean: ${price_data.mean():.2f}")
print(f"Median: ${price_data.median():.2f}")
print(f"Std: ${price_data.std():.2f}")
print(f"Min: ${price_data.min():.2f}")
print(f"Max: ${price_data.max():.2f}")

## 3. Category Analysis

In [ ]:
# Extract and analyze categories
all_categories = []

for categories_str in df['categories_clean'].dropna():
    if isinstance(categories_str, str) and categories_str.strip():
        try:
            categories = json.loads(categories_str.replace("'", '"'))
            if isinstance(categories, list):
                all_categories.extend(categories)
            else:
                all_categories.append(str(categories))
        except:
            all_categories.append(categories_str)

category_counts = pd.Series(all_categories).value_counts()

# Plot top categories
plt.figure(figsize=(12, 8))
top_categories = category_counts.head(20)
plt.barh(range(len(top_categories)), top_categories.values, color='lightblue')
plt.yticks(range(len(top_categories)), top_categories.index)
plt.xlabel('Number of Products')
plt.title('Top 20 Product Categories')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(f"Total unique categories: {len(category_counts)}")
print(f"\nTop 10 categories:")
print(category_counts.head(10))

## 4. Text Data Preprocessing

In [ ]:
def create_combined_text(row):
    """Create combined text for embedding from product features"""
    parts = []
    
    # Title (most important)
    if pd.notna(row.get('title')):
        parts.append(str(row['title']))
    
    # Description
    if pd.notna(row.get('description')):
        parts.append(str(row['description']))
    
    # Categories
    if pd.notna(row.get('categories_clean')):
        try:
            categories = json.loads(str(row['categories_clean']).replace("'", '"'))
            if isinstance(categories, list):
                parts.extend(categories)
            else:
                parts.append(str(categories))
        except:
            parts.append(str(row['categories_clean']))
    
    # Material
    if pd.notna(row.get('material_norm')):
        parts.append(f"material: {row['material_norm']}")
    
    # Brand
    if pd.notna(row.get('brand_norm')):
        parts.append(f"brand: {row['brand_norm']}")
    
    # Color
    if pd.notna(row.get('color_norm')):
        parts.append(f"color: {row['color_norm']}")
    
    return ' '.join(parts)

# Create combined text
df['combined_text'] = df.apply(create_combined_text, axis=1)

# Analyze text lengths
text_lengths = df['combined_text'].str.len()

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.hist(text_lengths, bins=50, alpha=0.7, color='lightcoral')
plt.xlabel('Text Length (characters)')
plt.ylabel('Frequency')
plt.title('Distribution of Combined Text Lengths')

plt.subplot(1, 2, 2)
word_counts = df['combined_text'].str.split().str.len()
plt.hist(word_counts, bins=50, alpha=0.7, color='lightgreen')
plt.xlabel('Word Count')
plt.ylabel('Frequency')
plt.title('Distribution of Word Counts')

plt.tight_layout()
plt.show()

print(f"Text length statistics:")
print(f"Mean characters: {text_lengths.mean():.1f}")
print(f"Mean words: {word_counts.mean():.1f}")
print(f"Max characters: {text_lengths.max()}")
print(f"Max words: {word_counts.max()}")

## 5. Embedding Model Training

In [ ]:
# Load pre-trained sentence transformer model
model_name = 'all-MiniLM-L6-v2'
model = SentenceTransformer(model_name)

print(f"Loaded model: {model_name}")
print(f"Model max sequence length: {model.max_seq_length}")

# Sample a subset for embedding generation (for demo purposes)
sample_size = min(1000, len(df))
sample_df = df.sample(n=sample_size, random_state=42)

print(f"Generating embeddings for {len(sample_df)} products...")
sample_texts = sample_df['combined_text'].tolist()

# Generate embeddings
embeddings = model.encode(sample_texts, convert_to_tensor=False, show_progress_bar=True)

print(f"Generated embeddings shape: {embeddings.shape}")
print(f"Embedding dimension: {embeddings.shape[1]}")

## 6. Similarity Analysis

In [ ]:
# Calculate pairwise similarities
similarity_matrix = cosine_similarity(embeddings)

# Analyze similarity distribution
# Get upper triangle (excluding diagonal)
upper_tri_indices = np.triu_indices_from(similarity_matrix, k=1)
similarities = similarity_matrix[upper_tri_indices]

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.hist(similarities, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
plt.xlabel('Cosine Similarity')
plt.ylabel('Frequency')
plt.title('Distribution of Pairwise Similarities')

plt.subplot(1, 2, 2)
plt.imshow(similarity_matrix[:50, :50], cmap='viridis')
plt.colorbar()
plt.title('Similarity Matrix (First 50x50)')
plt.xlabel('Product Index')
plt.ylabel('Product Index')

plt.tight_layout()
plt.show()

print(f"Similarity statistics:")
print(f"Mean similarity: {similarities.mean():.3f}")
print(f"Std similarity: {similarities.std():.3f}")
print(f"Min similarity: {similarities.min():.3f}")
print(f"Max similarity: {similarities.max():.3f}")

## 7. Recommendation Testing

In [ ]:
def get_recommendations(query, top_k=5):
    """Get product recommendations for a query"""
    # Generate query embedding
    query_embedding = model.encode([query], convert_to_tensor=False)
    
    # Calculate similarities
    query_similarities = cosine_similarity(query_embedding, embeddings)[0]
    
    # Get top-k indices
    top_indices = np.argsort(query_similarities)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        product = sample_df.iloc[idx]
        results.append({
            'title': product['title'],
            'price': product.get('price_num', 'N/A'),
            'similarity': query_similarities[idx],
            'categories': product.get('categories_clean', 'N/A')
        })
    
    return results

# Test with various queries
test_queries = [
    "comfortable office chair",
    "modern dining table",
    "cozy bedroom furniture",
    "outdoor patio set",
    "wooden bookshelf"
]

for query in test_queries:
    print(f"\n🔍 Query: '{query}'")
    print("-" * 50)
    
    recommendations = get_recommendations(query, top_k=3)
    
    for i, rec in enumerate(recommendations, 1):
        print(f"{i}. {rec['title'][:60]}...")
        print(f"   Price: ${rec['price']} | Similarity: {rec['similarity']:.3f}")
        print(f"   Categories: {str(rec['categories'])[:50]}...\n")

## 8. Model Evaluation

In [ ]:
# Evaluate model performance
def evaluate_recommendations(test_queries, top_k=10):
    """Evaluate recommendation quality"""
    results = []
    
    for query in test_queries:
        recommendations = get_recommendations(query, top_k=top_k)
        
        # Calculate metrics
        similarities = [rec['similarity'] for rec in recommendations]
        avg_similarity = np.mean(similarities)
        similarity_std = np.std(similarities)
        
        results.append({
            'query': query,
            'avg_similarity': avg_similarity,
            'similarity_std': similarity_std,
            'top_similarity': max(similarities),
            'min_similarity': min(similarities)
        })
    
    return pd.DataFrame(results)

# Evaluate
evaluation_results = evaluate_recommendations(test_queries)
print("Evaluation Results:")
print(evaluation_results)

# Plot evaluation metrics
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.bar(range(len(test_queries)), evaluation_results['avg_similarity'])
plt.xticks(range(len(test_queries)), [q[:20] + '...' for q in test_queries], rotation=45)
plt.ylabel('Average Similarity')
plt.title('Average Similarity by Query')

plt.subplot(2, 2, 2)
plt.bar(range(len(test_queries)), evaluation_results['similarity_std'])
plt.xticks(range(len(test_queries)), [q[:20] + '...' for q in test_queries], rotation=45)
plt.ylabel('Similarity Std Dev')
plt.title('Similarity Standard Deviation by Query')

plt.subplot(2, 2, 3)
plt.bar(range(len(test_queries)), evaluation_results['top_similarity'])
plt.xticks(range(len(test_queries)), [q[:20] + '...' for q in test_queries], rotation=45)
plt.ylabel('Top Similarity')
plt.title('Highest Similarity by Query')

plt.subplot(2, 2, 4)
plt.boxplot([get_recommendations(q, 20)[i]['similarity'] for q in test_queries for i in range(20)])
plt.ylabel('Similarity Score')
plt.title('Overall Similarity Distribution')

plt.tight_layout()
plt.show()

## 9. Conclusions and Next Steps

In [ ]:
print("📊 Model Training Summary")
print("=" * 50)
print(f"• Dataset size: {len(df):,} products")
print(f"• Sample size used: {len(sample_df):,} products")
print(f"• Embedding model: {model_name}")
print(f"• Embedding dimension: {embeddings.shape[1]}")
print(f"• Average similarity: {similarities.mean():.3f}")
print(f"• Unique categories: {len(category_counts)}")

print("\n🎯 Key Insights")
print("=" * 50)
print("• Semantic search works well for furniture queries")
print("• Price distribution is right-skewed with many affordable items")
print("• Category diversity provides good product coverage")
print("• Text embeddings capture product features effectively")

print("\n🚀 Next Steps")
print("=" * 50)
print("• Fine-tune embeddings on domain-specific data")
print("• Implement user behavior tracking for personalization")
print("• Add hybrid filtering (content + collaborative)")
print("• Optimize for production deployment")
print("• A/B test different recommendation strategies")